# Week 19, Day 2: Quicksort

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w019d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
items = [33, 10, 55, 71, 29, 3, 18, 42]
pivot = items[-1]
smaller = [x for x in items[:-1] if x < pivot]
bigger = [x for x in items[:-1] if x >= pivot]
print(smaller, pivot, bigger)

In [ ]:
def quicksort(items):
    if len(items) <= 1:
        return items[:]
    pivot = items[-1]
    rest = items[:-1]
    smaller = [x for x in rest if x < pivot]
    bigger = [x for x in rest if x >= pivot]
    return quicksort(smaller) + [pivot] + quicksort(bigger)

print(quicksort([33, 10, 55, 71, 29, 3, 18, 42]))

In [ ]:
def partition(items, lo, hi):
    pivot = items[hi]
    i = lo                                  # items[lo:i] are all < pivot
    for j in range(lo, hi):
        if items[j] < pivot:
            items[i], items[j] = items[j], items[i]
            i += 1
    items[i], items[hi] = items[hi], items[i]   # the pivot goes between the sides
    return i                                # its final position

def quicksort(items, lo=0, hi=None):
    if hi is None:
        hi = len(items) - 1
    if lo < hi:
        p = partition(items, lo, hi)
        quicksort(items, lo, p - 1)
        quicksort(items, p + 1, hi)

data = [33, 10, 55, 71, 29, 3, 18, 42]
quicksort(data)
print(data)

*Interactive figure on the website: Quicksort: after each partition, the pivot is in its final place and the two sides are sorted separately. Try sorted data and count the comparisons..*

In [ ]:
import random

def quicksort_count(items):
    comparisons = 0
    def sort(lo, hi):
        nonlocal comparisons
        while lo < hi:                      # loop on one side to save stack depth
            pivot, i = items[hi], lo
            for j in range(lo, hi):
                comparisons += 1
                if items[j] < pivot:
                    items[i], items[j] = items[j], items[i]
                    i += 1
            items[i], items[hi] = items[hi], items[i]
            sort(lo, i - 1)
            lo = i + 1
    sort(0, len(items) - 1)
    return comparisons

n = 800
print("random:", quicksort_count(random.Random(1).sample(range(n), n)))
print("reversed:", quicksort_count(list(range(n, 0, -1))))
print("n(n-1)/2:", n * (n - 1) // 2)

In [ ]:
import random

def partition(items, lo, hi):
    r = random.randint(lo, hi)
    items[r], items[hi] = items[hi], items[r]    # a random item becomes the pivot
    pivot = items[hi]
    i = lo
    for j in range(lo, hi):
        if items[j] < pivot:
            items[i], items[j] = items[j], items[i]
            i += 1
    items[i], items[hi] = items[hi], items[i]
    return i

def quicksort(items, lo=0, hi=None):
    if hi is None:
        hi = len(items) - 1
    if lo < hi:
        p = partition(items, lo, hi)
        quicksort(items, lo, p - 1)
        quicksort(items, p + 1, hi)

data = list(range(3000))          # sorted input: deadly for a last-item pivot
quicksort(data)
print(data[:5], data[-5:])

In [ ]:
def partition3(items, lo, hi):
    pivot = items[lo]
    lt, i, gt = lo, lo, hi
    # items[lo:lt] < pivot, items[lt:i] == pivot, items[gt+1:hi+1] > pivot
    while i <= gt:
        if items[i] < pivot:
            items[lt], items[i] = items[i], items[lt]
            lt += 1
            i += 1
        elif items[i] > pivot:
            items[i], items[gt] = items[gt], items[i]
            gt -= 1
        else:
            i += 1
    return lt, gt

def quicksort3(items, lo=0, hi=None):
    if hi is None:
        hi = len(items) - 1
    if lo < hi:
        lt, gt = partition3(items, lo, hi)
        quicksort3(items, lo, lt - 1)
        quicksort3(items, gt + 1, hi)

marks = [3, 1, 2, 3, 3, 1, 2, 2, 3, 1]
quicksort3(marks)
print(marks)

In [ ]:
import random

def quickselect(items, k):
    """The kth smallest item (k = 0 is the smallest). Changes the order of items."""
    lo, hi = 0, len(items) - 1
    while True:
        if lo == hi:
            return items[lo]
        r = random.randint(lo, hi)
        items[r], items[hi] = items[hi], items[r]
        pivot, i = items[hi], lo
        for j in range(lo, hi):
            if items[j] < pivot:
                items[i], items[j] = items[j], items[i]
                i += 1
        items[i], items[hi] = items[hi], items[i]
        if k == i:
            return items[i]
        if k < i:
            hi = i - 1
        else:
            lo = i + 1

data = [33, 10, 55, 71, 29, 3, 18, 42]
print(quickselect(data[:], 0), quickselect(data[:], 3), quickselect(data[:], 7))

> **Key idea.** Quicksort: partition around a pivot, then sort each side. O(n log n) on average, O(n²) when pivots split badly; a random pivot makes bad splits vanishingly unlikely. In place, not stable. Quickselect uses one partition per round to find the kth smallest in O(n) on average.

### Exercise: Quicksort with comprehensions

Write `quicksort(items)` returning a new sorted list using the comprehension version: pivot on the **first** item, and split the rest into smaller-than and not-smaller-than. Don't use `sort` or `sorted`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_values():\n    \"several\"\n    for data in ([33, 10, 55, 71, 29, 3, 18, 42], [], [1], [2, 1], [4, 4, 4], [\"kiwi\", \"fig\", \"apple\"]):\n        before = list(data)\n        assert quicksort(data) == sorted(data) and data == before\n    rng = random.Random(3)\n    data = [rng.randint(0, 10_000) for _ in range(5000)]\n    assert quicksort(data) == sorted(data)\n    import re\n    assert not re.search(r\"\\bsorted\\(|\\.sort\\(\", source)")

### Exercise: Partition in place

Write `partition(items, lo, hi)` using the Lomuto scheme with `items[hi]` as the pivot. It rearranges `items[lo]` to `items[hi]` in place and returns the pivot's final position. Then `quicksort_in_place(items)` that sorts a list in place using it.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_partition():\n    \"partition\"\n    xs = [33, 10, 55, 71, 29, 3, 18, 42]\n    p = partition(xs, 0, 7)\n    assert p == 5 and xs[p] == 42\n    assert all(x < 42 for x in xs[:p]) and all(x >= 42 for x in xs[p + 1:])\n    ys = [9, 1, 8, 2, 5]\n    p = partition(ys, 1, 3)\n    assert ys[0] == 9 and ys[4] == 5 and ys[p] == 2 and p == 2\ndef test_sort():\n    \"sort\"\n    rng = random.Random(1)\n    for _ in range(50):\n        data = [rng.randint(0, 50) for _ in range(rng.randint(0, 60))]\n        copy = data[:]\n        quicksort_in_place(copy)\n        assert copy == sorted(data)")

### Exercise: How deep?

Write `max_depth(items)` that runs in-place quicksort (last-item pivot) on a copy and returns the deepest level of recursion reached (the first call is depth 1; a call with fewer than 2 items doesn't count). Then set `sorted_100` to the depth for `list(range(100))` and say whether random data of 100 items goes deeper (`random_deeper`, `True` or `False`).

In [ ]:
sorted_100 = ___
random_deeper = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_values():\n    \"several\"\n    assert max_depth([]) == 0 and max_depth([1]) == 0 and max_depth([2, 1]) == 1\n    assert max_depth(list(range(100))) == 99 == sorted_100\n    d = max_depth(random.Random(5).sample(range(100), 100))\n    assert d < 30 and random_deeper is False")

### Exercise: Three colours

A list holds only the strings `"red"`, `"white"` and `"blue"`. Write `flag(items)` that rearranges it in place into all reds, then whites, then blues, in **one pass** with O(1) extra space, using three-way partitioning with `"white"` playing the pivot. Don't count, sort, or build new lists.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_values():\n    \"several\"\n    xs = [\"blue\", \"white\", \"red\", \"white\", \"red\", \"blue\", \"red\"]\n    assert flag(xs) is None and xs == [\"red\"] * 3 + [\"white\"] * 2 + [\"blue\"] * 2\n    ys = []\n    flag(ys)\n    assert ys == []\n    rng = random.Random(2)\n    zs = [rng.choice([\"red\", \"white\", \"blue\"]) for _ in range(100_000)]\n    want = [\"red\"] * zs.count(\"red\") + [\"white\"] * zs.count(\"white\") + [\"blue\"] * zs.count(\"blue\")\n    flag(zs)\n    assert zs == want\n    assert \"count(\" not in source and \"sort\" not in source")

### Exercise: The median in O(n)

Write `median(items)` for a non-empty list of numbers with an odd length, using quickselect (random pivot) on a **copy**, without sorting. It must handle 200,001 numbers quickly.

In [ ]:
import random

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random as rnd\ndef test_values():\n    \"several\"\n    assert median([5]) == 5 and median([3, 1, 2]) == 2 and median([9, 1, 8, 2, 7]) == 7\n    data = [4, 4, 1, 4, 9]\n    assert median(data) == 4 and data == [4, 4, 1, 4, 9]\n    r = rnd.Random(9)\n    big = [r.randint(0, 10 ** 9) for _ in range(200_001)]\n    assert median(big) == sorted(big)[100_000]\n    import re\n    assert not re.search(r\"\\bsorted\\(|\\.sort\\(\", source)")

### Bug hunt: Sorted is too much

This quicksort crashes with `RecursionError` on a sorted list of 3,000 numbers, because the last-item pivot splits it as badly as possible. Fix it with a random pivot, so it prints `True`.

In [ ]:
import random

def quicksort(items, lo, hi):
    if lo < hi:
        pivot, i = items[hi], lo
        for j in range(lo, hi):
            if items[j] < pivot:
                items[i], items[j] = items[j], items[i]
                i += 1
        items[i], items[hi] = items[hi], items[i]
        quicksort(items, lo, i - 1)
        quicksort(items, i + 1, hi)

data = list(range(3000))
quicksort(data, 0, len(data) - 1)
print(data == list(range(3000)))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"True\"\n    expect_output(\"True\")")

### Bug hunt: The pivot vanishes

This comprehension quicksort loses items: `[3, 1, 3, 2]` comes back as `[1, 2, 3]`. Fix it so it prints `[1, 2, 3, 3]`.

In [ ]:
def quicksort(items):
    if len(items) <= 1:
        return items[:]
    pivot = items[0]
    smaller = [x for x in items if x < pivot]
    bigger = [x for x in items if x > pivot]
    return quicksort(smaller) + [pivot] + quicksort(bigger)

print(quicksort([3, 1, 3, 2]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"all items\"\n    expect_output(\"[1, 2, 3, 3]\")")

### Puzzle: Lomuto partition

Arrange the lines for `partition`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Work these out without running them.

In [ ]:
a = ___   # the list after partition([4, 7, 1, 6, 3], 0, 4) (Lomuto, pivot is the last item)
b = ___   # the position partition returns for that call
c = ___   # comparisons quicksort (last-item pivot) makes on [1, 2, 3, 4, 5]
d = ___   # True or False: quicksort is stable

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == [1, 3, 4, 6, 7], \"a\"\n    assert b == 1 and c == 10 and d is False")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a hybrid sort

Real sorting libraries mix algorithms. Write `hybrid_sort(items, cutoff=16)` that sorts a list in place with quicksort (random pivot), but switches to **insertion sort** on any part with fewer than `cutoff` items, since insertion sort is faster on tiny lists. Also, to keep the stack shallow even with bad luck, recurse only into the **smaller** side and loop on the larger side, so the recursion is at most about log₂ n deep. Return the deepest recursion depth reached (the first call is depth 1).

It must sort 100,000 numbers, and sorted, reversed and all-equal lists of 20,000, without hitting the recursion limit.

In [ ]:
import random

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random as rnd\ndef test_values():\n    \"sorts\"\n    r = rnd.Random(4)\n    for n in (0, 1, 2, 15, 16, 17, 100, 1000):\n        data = [r.randint(0, 50) for _ in range(n)]\n        copy = data[:]\n        hybrid_sort(copy)\n        assert copy == sorted(data), n\ndef test_big():\n    \"big and awkward\"\n    r = rnd.Random(5)\n    data = [r.randint(0, 10 ** 9) for _ in range(100_000)]\n    copy = data[:]\n    depth = hybrid_sort(copy)\n    assert copy == sorted(data) and depth <= 18\n    for data in (list(range(20_000)), list(range(20_000, 0, -1))):\n        copy = data[:]\n        hybrid_sort(copy)\n        assert copy == sorted(data)\n    same = [7] * 20_000\n    hybrid_sort(same)\n    assert same == [7] * 20_000")

## Warm-up

Whatever order the two sides are in, the pivot 42 is now in exactly the place it will be when the whole list is sorted: everything smaller is to its left, everything bigger to its right. Sort the two sides, and you're done.

## The idea

**Quicksort:**

1. If the list has 0 or 1 items, it's sorted.
2. **Partition:** pick a pivot; put the items smaller than it on one side and the rest on the other.
3. Sort each side recursively. The answer is the sorted smaller side, then the pivot, then the sorted bigger side.

Compare with merge sort. Merge sort splits **blindly** in the middle and does its work **after** the recursion, in the merge. Quicksort does its work **before** the recursion, in the partition, and the split point depends on the pivot. After the recursive calls there's nothing left to do.

This version is clear but builds new lists at every step. Real quicksort works **in place**, which is where its speed comes from.

## Partitioning in place

The **Lomuto partition** sorts positions `lo` to `hi` of one list, using the last item as the pivot. It keeps a boundary `i`: everything before position `i` is smaller than the pivot. It walks `j` across the range; each time it finds a smaller item, it swaps it to position `i` and moves the boundary along. Finally it swaps the pivot into position `i`, between the two sides.

Watch it. The red bar is the pivot, and each green bar is a pivot that has reached its final place:

In place, quicksort needs no buffer: only the call stack, O(log n) deep when the splits are even. It's **not stable**: the long-distance swaps in the partition can reorder equal items.

## Good and bad pivots

How fast quicksort is depends entirely on how evenly the pivots split.

- **Good splits:** if each pivot lands near the middle, the sizes halve each level: about log₂ n levels with n work each, **O(n log n)**. On random data the splits are usually good enough, and the average is about 1.39 n log₂ n comparisons.
- **Bad splits:** if each pivot is the smallest or largest item, one side is empty and the other has everything but the pivot: n levels with n, n − 1, n − 2, ... work: n(n − 1)/2 comparisons, **O(n²)**, and n levels of recursion.

With the last item as the pivot, the worst case happens on data that's **already sorted** (or reversed), which is very common in practice:

The fix is to choose the pivot so that no particular input is reliably bad. The simplest: a **random pivot**. Swap a randomly chosen item into the last place before partitioning:

Now **no input** is reliably bad: the worst case can still happen, but only by extreme bad luck, with a chance smaller than winning the lottery many times in a row. Another common choice is the **median of three**: the middle value of the first, middle and last items.

| | Merge sort | Quicksort (random pivot) |
|---|---|---|
| Average | O(n log n) | O(n log n), usually faster in practice |
| Worst | O(n log n) | O(n²), but vanishingly unlikely |
| Extra space | O(n) | O(log n) stack, on average |
| Stable | yes | no |
| Work done | after the recursion (merge) | before the recursion (partition) |

In languages like C, quicksort is usually the fastest general sort, because it works in place and moves through memory in order, which modern processors love. Many standard libraries use a quicksort variant for numbers.

## Repeated values

What if the list is full of equal values, say a million exam marks out of 100? With the partition above, every item equal to the pivot goes to the "bigger" side, and a list of all-equal values becomes the worst case again.

**Three-way partitioning** splits into three parts: smaller than the pivot, **equal** to it, and bigger. The equal part is already in place and never touched again. This is Edsger Dijkstra's "Dutch national flag" problem (three colours, like the flag's three stripes):

Notice that when an item goes to the "bigger" end, `i` does **not** move on: the item swapped in from position `gt` hasn't been looked at yet.

## Quickselect

Often you don't need the whole list sorted, only one item: the median, or the 10th smallest. Partition once, and you know the pivot's final position `p`. If that's the position you want, you're done; if not, the answer is on **one** side only, so recurse into that side and ignore the other.

Each round keeps about half of what's left, on average: n + n/2 + n/4 + ... is about 2n, so quickselect is **O(n) on average**, faster than sorting. It's how you'd find a median in one pass over a huge list.

## Practice

## Quiz

## Daily challenge

## Before you go

Sort a shuffled deck by quicksort: pick the top card as the pivot, deal the rest into a "lower" pile and a "higher" pile, and repeat on each pile. Then try it again on a deck that's already in order, and feel why the first card is a terrible pivot. Tomorrow: counting sort, which sorts without comparing anything at all.